# 01 — Bayes' rule, from counts

*Module 02 · Naive Bayes — notebook 1 of 5*

In module 01 you classified by **distance**: a point took the label of its nearest neighbours. Here
we open a different door — classifying by **probability**. Given a penguin's measurement, we ask a
sharp question: *how probable is each species?* — and answer it with one of the oldest, most useful
rules in statistics, **Bayes' rule**.

The good news: we need nothing but **counting**. No equation to solve, no curve to fit — count
penguins, turn counts into probabilities, and combine them with Bayes. You will build the whole thing
by hand, on a single feature.

**Prerequisites:** module 01 (k-NN — the idea of a classifier, now rebuilt on probability); module 00
notebook 06 (class frequencies and the majority baseline).

**What you will be able to do**

- Turn counts into a **prior**, a **likelihood**, and a **posterior** — by hand.
- State **Bayes' rule** and name its four terms (prior, likelihood, posterior, evidence).
- Predict a class by taking the **argmax** of the posterior.
- Recognise the **zero-frequency** trap — a probability of exactly zero from an unseen combination —
  and name its fix.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ml_course import datasets, viz
from ml_course.colors import CLASS_CYCLE, COLORS

# Fix the seed so every run tells the same story (a good habit, even with no randomness here).
np.random.seed(0)
viz.use_course_style()

# The familiar binary penguins subset: Adelie vs Gentoo, two measurements per bird.
df = datasets.load_penguins()
print(f"rows: {df.shape[0]}   columns: {list(df.columns)}")
print(df["species"].value_counts())
df.head()

## A recap, and a change of question

One idea from module 00 carries this whole chapter: a **probability is a relative frequency** — count
how often something happens, divide by the total. If 151 of 274 penguins are Adélie, then
P(Adélie) = 151 / 274.

Module 01 asked *which training points are nearest?* This chapter asks something else: **given what we
measured, how probable is each species?** We have two species (Adélie, Gentoo) and, to start, a single
measurement — the bill length. Our target is the probability of a species *given* a measurement,
written P(species | measurement). Bayes' rule is the bridge that takes us there, and every piece of it
is a count.

## The prior: belief before the measurement

Before we look at a single bill, what is our best guess for a penguin's species? The honest answer is
the **base rate** — how common each species is in the data. That belief, held *before* seeing any
measurement, is the **prior**, written P(species).

In [ ]:
# The prior is a relative frequency: each species' share of all penguins.
prior = df["species"].value_counts(normalize=True).sort_index()
print(prior.round(4))

species = list(prior.index)
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(
    species,
    prior.to_numpy(),
    color=[CLASS_CYCLE[i] for i in range(len(species))],
    edgecolor=COLORS["text"],
    linewidth=0.6,
)
for i, p in enumerate(prior.to_numpy()):
    ax.text(i, p, f"{p:.3f}", ha="center", va="bottom", color=COLORS["text"])
ax.set_ylabel("P(species)")
ax.set_title("The prior — belief before any measurement")
plt.show()

**Read the figure.** With no measurement in hand, our best guess is the base rate: Adélie is a touch
more common (P = 0.551) than Gentoo (P = 0.449). These two numbers are our starting belief — a
baseline the measurement will sharpen. They sum to 1, because every penguin is one species or the
other.

## One feature, turned into counts

The measurement we will use is the **bill length**. There is a snag: bill length is a continuous number
(32.1 mm, 45.8 mm, …), and you cannot count how many penguins have a bill of *exactly* 45.8 mm — almost
no two birds share an exact value. So we **bin** it: group bill lengths into three readable categories
— short, medium, long — and count within each. Binning is a teaching device that lets us count; in
notebook 3 we replace it with a smooth curve that needs no bins.

In [ ]:
# Three readable bins for bill length (mm); edges chosen to be round and legible.
bin_edges = [30, 42, 48, 62]
bin_labels = ["short (30-42)", "medium (42-48)", "long (48-62)"]
df = df.assign(
    bill_bin=pd.cut(df["bill_length_mm"], bins=bin_edges, labels=bin_labels, include_lowest=True)
)

# Count penguins in each (species, bin) cell: the contingency table.
counts = pd.crosstab(df["species"], df["bill_bin"])
print(counts)

**Read the figure.** Read the table as counts of penguins. Adélie birds cluster in the **short** bin
(135 of them), a few reach **medium** (16), and **none** are **long** (0). Gentoo lean the other way:
mostly **medium** (67) and **long** (53), with only 3 in **short**. Bill length already separates the
two species strongly. Keep an eye on that **0** in the Adélie / long cell — it has something to teach
us shortly.

## The likelihood: what each species' bills look like

The contingency table counts penguins in each species–bin combination. The **likelihood** asks a
**conditional** question — a probability computed *within* a restricted group: *given that a penguin is
an Adélie, how probable is each bill bin?* That is P(bin | species), where the vertical bar reads
"given". We read it straight off the table by normalising **each species' row** to sum to 1.

In [ ]:
# Likelihood P(bin | species): normalise each species' row to sum to 1.
likelihood = counts.div(counts.sum(axis=1), axis=0)
print(likelihood.round(4))

# A grouped bar: one cluster per bin, one bar per species.
ax = likelihood.T.plot(
    kind="bar",
    figsize=(7, 4.5),
    color=[CLASS_CYCLE[0], CLASS_CYCLE[1]],
    edgecolor=COLORS["text"],
    linewidth=0.6,
    rot=0,
)
ax.set_xlabel("bill-length bin")
ax.set_ylabel("P(bin | species)")
ax.set_title("Likelihood — bill-length distribution within each species")
ax.legend(title="species")
plt.show()

**Read the figure.** Each species' bars sum to 1 — this is how *that species'* bills distribute
across the bins. (They sum to 1 because the three bins cover every Adélie's bill; that is a feature of
this discrete, binned case, not of likelihoods in general — in notebook 3 a smooth likelihood will
not.) Adélie are overwhelmingly short (P(short | Adélie) = 0.894) and **never long**
(P(long | Adélie) = 0.000). Gentoo spread across medium (0.545) and long (0.431). That single zero is
the **zero-frequency** trap: because we never *observed* a long-billed Adélie, the likelihood declares
one **impossible** — P = 0. That is too strong a claim to draw from a finite sample, and it will spread
into the posterior. Notebook 4 fixes it with **smoothing** (gently adding to every count).

## Bayes' rule: from P(bin | species) to P(species | bin)

We can measure P(bin | species) by counting — but what we actually want is the reverse,
P(species | bin): the probability of a species *given* the bin we observed. **Bayes' rule** turns one
into the other:

$$ \underbrace{P(\text{species} \mid \text{bin})}_{\text{posterior}}
= \frac{\overbrace{P(\text{bin} \mid \text{species})}^{\text{likelihood}}\;
\overbrace{P(\text{species})}^{\text{prior}}}{\underbrace{P(\text{bin})}_{\text{evidence}}} $$

Four terms, each with a job:

- **prior** P(species) — belief before the measurement;
- **likelihood** P(bin | species) — how well the species explains the measurement;
- **posterior** P(species | bin) — belief *after* the measurement (what we want);
- **evidence** P(bin) — how common the measurement is overall; the **normaliser** that makes the
  posteriors across species sum to 1.

In [ ]:
# Bayes, by hand. Numerator: likelihood x prior for every (species, bin).
joint = likelihood.mul(prior, axis=0)        # P(bin | species) * P(species)

# Evidence: how common each bin is overall = sum of the numerator over species.
evidence = joint.sum(axis=0)                 # P(bin)

# Posterior: divide by the evidence so each bin's column sums to 1.
posterior = joint.div(evidence, axis=1)      # P(species | bin)
print("evidence P(bin):")
print(evidence.round(4))
print("\nposterior P(species | bin):")
print(posterior.round(4))

ax = posterior.T.plot(
    kind="bar",
    stacked=True,
    figsize=(7, 4.5),
    color=[CLASS_CYCLE[0], CLASS_CYCLE[1]],
    edgecolor=COLORS["text"],
    linewidth=0.6,
    rot=0,
)
ax.set_xlabel("bill-length bin")
ax.set_ylabel("P(species | bin)")
ax.set_title("Posterior — belief after seeing the bill bin")
ax.legend(title="species", loc="center left", bbox_to_anchor=(1.0, 0.5))
plt.show()

**Read the figure.** Each bar is one bin, split into the two species' posterior probabilities — and
each bar sums to 1. A **short** bill points firmly to Adélie (P = 0.978). A **medium** bill leans
Gentoo (0.807) but leaves real room for doubt — the posterior is a *degree of belief*, not a verdict,
and here it honestly reports its uncertainty. A **long** bill gives Gentoo a posterior of exactly
**1.000** — and that crispness is the zero-frequency trap resurfacing: it is 1 only because no Adélie
was ever seen with a long bill, not because it is truly impossible.

In [ ]:
# Predict each bin's species: the one with the largest posterior.
prediction = posterior.idxmax(axis=0)
for b in posterior.columns:
    print(f"bin {b:>15} -> predict {prediction[b]:7}  (posterior {posterior[b].max():.3f})")

# The evidence cancels under argmax: P(bin) divides both species equally within a bin,
# so ranking by the (unnormalised) numerator gives the SAME winner as the posterior.
same = joint.idxmax(axis=0).equals(posterior.idxmax(axis=0))
print(f"\nargmax of (likelihood x prior) matches argmax of posterior: {same}")

**What just happened — and a one-line shortcut.** Reading off the largest posterior in each bin
gives a classifier: short → Adélie, medium → Gentoo, long → Gentoo. And notice the last line: because
the **evidence** P(bin) divides both species equally within a bin, it never changes *which* species is
largest. To *decide*, we can compare prior × likelihood directly and skip the normalisation — the
evidence matters for the probability, not for the choice.

**Vocabulary** (the running Naive-Bayes glossary starts here)

- **prior** P(y) — belief in each class before any measurement (the base rate).
- **likelihood** P(x | y) — how probable the measurement is under each class.
- **posterior** P(y | x) — belief in each class after the measurement; what we predict from.
- **evidence** P(x) — how probable the measurement is overall; the normaliser; cancels under argmax.
- **argmax** — pick the class with the largest posterior.
- **zero-frequency** — a likelihood (and posterior) of exactly 0 from a combination never seen in the
  data.

## Your turn

Make the rule yours. Reason first; a little code helps for (b) and (c).

1. **Shift the prior.** Suppose Adélie and Gentoo were equally common — set P(species) = 0.5 each.
   Recompute the posterior for the **medium** bin by hand. Does the winner change? Which term moved it?
2. **Re-bin.** Rebuild the contingency table with four bins instead of three (or with different edges)
   and recompute the posteriors. Where does the prediction stay firm, and where does it wobble?
3. **(Harder) Heal the zero.** The long-bin posterior for Gentoo is exactly 1.000 because no Adélie was
   ever seen there. Add 1 to *every* count in the contingency table, recompute the likelihood and
   posterior, and describe what happens to that 1.000. This is **Laplace smoothing** — we formalise it
   in notebook 4.

## What you built

You built a classifier from nothing but counts. You turned class frequencies into a **prior**, a
contingency table into a **likelihood**, and combined them with **Bayes' rule** into a **posterior** —
then predicted by **argmax**. Along the way you met the **zero-frequency** trap, saw why a posterior of
exactly 0 or 1 from a finite sample is overconfident, and named its cure.

You can now:

- compute a prior, a likelihood, and a posterior by hand;
- state Bayes' rule and explain each of its four terms;
- predict a class by the largest posterior, and explain why the evidence cancels in that choice;
- spot a zero-frequency probability and say what smoothing will do about it.

## Going further (optional)

Two doors this notebook opens, both walked through next:

- **More than one feature.** Real penguins carry several measurements. Combining them needs
  P(bill, flipper | species) — a *joint* likelihood. Notebook 2 makes the famous "naive" assumption
  that builds it from single-feature pieces, and asks honestly when that is fair.
- **No bins.** Binning threw away detail. Notebook 3 replaces the histogram with a smooth **Gaussian**
  curve fitted to each species — a likelihood for continuous features that needs no bins.

Neither is required to continue; they are the next two steps.

## References

- T. Bayes, R. Price (1763). *An essay towards solving a problem in the doctrine of chances.*
  Philosophical Transactions of the Royal Society, 53:370–418. DOI:
  [10.1098/rstl.1763.0053](https://doi.org/10.1098/rstl.1763.0053) — the origin of the rule.
- G. James, D. Witten, T. Hastie, R. Tibshirani (2021). *An Introduction to Statistical Learning*
  (§4.4.4, naive Bayes). DOI:
  [10.1007/978-1-0716-1418-1](https://doi.org/10.1007/978-1-0716-1418-1).

---

*Previous: Module 01 — k-Nearest Neighbours* · *Next: 02 — The "naive" assumption*